# Chapter 3 — The forecaster's toolbox (Python version)

*Study notes in R and Python · Forecasting: Principles and Practice (2nd ed.)*

Auto-generated from `ch03_notes.Rmd` by `python/rmd_to_ipynb.py` — the R code lives in the .Rmd file. Run this notebook from its own folder.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath("../python"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import fpp
import fpp_ch03 as ch3
plt.rcParams.update({"figure.figsize": (8, 4.2), "figure.dpi": 110})
pd.set_option("display.width", 160, "display.max_columns", 20)
%matplotlib inline

> **Acknowledgement.** These are my personal study notes for **Chapter 3** of
> *Forecasting: Principles and Practice* (2nd ed.) by **Rob J Hyndman and George
> Athanasopoulos** (OTexts, 2018, <https://otexts.com/fpp2/>). I follow the authors'
> teaching section by section; the ideas, structure, datasets and exercises are theirs.
> The explanations are summarised in my own words, and every code step is shown twice:
> **R** (as in the book) and **Python** (my translation).

# Learning goals

* Four **benchmark methods** — mean, naïve, seasonal naïve, drift — that every serious method must beat.
* **Adjustments and transformations** (calendar, population, inflation, Box-Cox) and **bias adjustment**.
* **Residual diagnostics**: what good residuals look like, portmanteau tests, `checkresiduals()`.
* **Accuracy**: training/test sets, MAE, RMSE, MAPE, sMAPE, MASE, **time series cross-validation**.
* **Prediction intervals**: one-step, multi-step, benchmark formulas, **bootstrap**, with transformations.

# 3.1 Some simple forecasting methods

Some very simple methods are surprisingly effective; they serve as **benchmarks**.

| Method | Forecast $\hat{y}_{T+h|T}$ | R |
|---|---|---|
| Average (mean) | $\bar{y} = (y_1 + \dots + y_T)/T$ | `meanf(y, h)` |
| Naïve | $y_T$ (last observation) — optimal for a random walk | `naive(y, h)` / `rwf(y, h)` |
| Seasonal naïve | $y_{T+h-m(k+1)}$: last observed value from the same season ($m$ = period, $k$ = integer part of $(h-1)/m$) | `snaive(y, h)` |
| Drift | $y_T + h\,\frac{y_T - y_1}{T-1}$: the line through the first and last observations, extended | `rwf(y, h, drift = TRUE)` |

In Python the same four methods are in `python/fpp.py` (`fpp.meanf`, `fpp.naive`, `fpp.snaive`,
`fpp.rwf(drift=True)`) and are tested to reproduce R's numbers exactly.

### Beer production: mean, naïve and seasonal naïve (Figure 3.1)

In [ ]:
ausbeer = fpp.load("ausbeer")
beer2 = fpp.window(ausbeer, start=1992, end=(2007, 4))
p_ = ch3.plot_forecasts(beer2, {"Mean": fpp.meanf(beer2, h=11), "Naive": fpp.naive(beer2, h=11),
                                "Seasonal naive": fpp.snaive(beer2, h=11)},
                        title="Forecasts for quarterly beer production", xlab="Year", ylab="Megalitres")
plt.show()

### Google stock: mean, naïve and drift (Figure 3.2)

In [ ]:
goog200 = fpp.load("goog200")
p_ = ch3.plot_forecasts(goog200, {"Mean": fpp.meanf(goog200, h=40), "Naive": fpp.rwf(goog200, h=40),
                                  "Drift": fpp.rwf(goog200, drift=True, h=40)},
                        title="Google stock (daily ending 6 Dec 2013)", xlab="Day", ylab="Closing Price (US$)")
plt.show()

Sometimes a benchmark is the best method available; usually it is the yardstick that a new
method has to beat — if it can't, it isn't worth using.

# 3.2 Transformations and adjustments

Adjusting the data first can **simplify the patterns** (remove known sources of variation,
make the pattern more consistent), and simpler patterns usually forecast better.
Four kinds: **calendar**, **population**, **inflation** and **mathematical** transformations.

## Calendar adjustments

Some seasonal variation is just calendar arithmetic, e.g. months have 28–31 days.
Dividing monthly milk production by the number of days gives **average daily production**,
whose seasonal pattern is much simpler. The same idea applies to sales per *trading day*.

### Monthly vs daily-average milk production (Figure 3.3)

In [ ]:
milk = fpp.load("milk")
dframe = pd.DataFrame({"Monthly": milk, "DailyAverage": milk / ch3.monthdays(milk)})
dframe.attrs = dict(milk.attrs)
p_ = fpp.autoplot(dframe, facets=True, title="Milk production per cow", xlab="Years")
plt.show()

## Population adjustments

Series affected by population size should be put **per capita** (per person, per thousand
people...). E.g. hospital beds per 1000 people shows whether provision really improved, or
whether the total rose only because the population grew.

## Inflation adjustments

Money values should be expressed in **constant dollars** using a price index $z_t$ (e.g. the CPI):
$x_t = y_t / z_t \times z_{2000}$ states each value in year-2000 dollars.

### Toy example of population and inflation adjustment

*(Illustration with made-up numbers — the book gives no dataset here.)*

In [ ]:
toy = pd.DataFrame({"beds": [2000, 2100, 2210, 2300, 2380], "pop": [400, 425, 452, 480, 510],
                    "price": [200, 206, 213, 221, 230], "cpi": [100, 102, 105, 109, 112]},
                   index=range(2014, 2019))
toy["beds_per_1000"] = (toy.beds / toy["pop"]).round(2)
toy["price_2014_dollars"] = (toy.price / toy.cpi * toy.cpi.iloc[0]).round(1)
toy[["beds", "beds_per_1000", "price", "price_2014_dollars"]]

Beds rise every year, but **beds per 1000 people fall** — population grew faster.
Prices rise, but in constant dollars the increase is much smaller.

## Mathematical transformations

When the variation grows (or shrinks) with the level, transform:

* **log**: $w_t = \log(y_t)$ — changes on the log scale are *relative* (percentage) changes;
  forecasts stay positive on the original scale.
* **power** transformations: square root, cube root, ...
* **Box-Cox family** (depends on $\lambda$):
$$w_t = \begin{cases} \log(y_t) & \lambda = 0 \\ (y_t^\lambda - 1)/\lambda & \text{otherwise.}\end{cases}$$

$\lambda = 1$: no substantive change (just shifted down by 1); $\lambda = 1/2$: square root plus
linear transformation; $\lambda = 0$: natural log; $\lambda = -1$: inverse plus 1.
A good $\lambda$ makes the seasonal variation about the same across the whole series.
`BoxCox.lambda()` chooses $\lambda$ automatically (Guerrero's method — replicated by
`fpp.box_cox_lambda()`).

### Box-Cox transformation of electricity production

In [ ]:
elec = fpp.load("elec")
lam = fpp.box_cox_lambda(elec)
print(round(lam, 4))
p_ = fpp.autoplot(fpp.box_cox(elec, lam), title=f"Box-Cox transformed elec, lambda = {lam:.3f}")
plt.show()

After forecasting on the transformed scale, **back-transform**:
$y_t = \exp(w_t)$ if $\lambda = 0$, otherwise $y_t = (\lambda w_t + 1)^{1/\lambda}$.

Features of power transformations:

* if any $y_t \le 0$, power transformations are not possible unless all values are first shifted
  to be positive (add a constant);
* simple values of $\lambda$ are easier to explain;
* results are relatively insensitive to the exact $\lambda$;
* often **no** transformation is needed at all;
* transformations usually affect **prediction intervals** much more than point forecasts.

## Bias adjustments

Back-transformed point forecasts are the **median** of the forecast distribution, not the
**mean**. If you need the mean (e.g. to add up regional forecasts), use the bias-adjusted
back-transformation:
$$y_t = \begin{cases} \exp(w_t)\left[1 + \tfrac{\sigma_h^2}{2}\right] & \lambda = 0\\
(\lambda w_t + 1)^{1/\lambda}\left[1 + \tfrac{\sigma_h^2 (1-\lambda)}{2(\lambda w_t + 1)^2}\right] & \text{otherwise,}\end{cases}$$
where $\sigma_h^2$ is the $h$-step forecast variance on the transformed scale. The larger the
variance, the bigger the difference between mean and median.

### Drift forecasts of egg prices on the log scale (Figure 3.4)

In [ ]:
eggs = fpp.load("eggs")
fc = fpp.rwf(eggs, drift=True, lambda_=0, h=50, level=(80,))
fc2 = fpp.rwf(eggs, drift=True, lambda_=0, h=50, level=(80,), biasadj=True)
ax = ch3.plot_forecasts(eggs, {"Simple back transformation": fc}, pi=True)
p_ = ax.plot(fpp.ts_time(eggs)[-1] + np.arange(1, 51), fc2.mean.to_numpy(), color="C1", label="Bias adjusted")
p_ = ax.legend(title="Forecast")
plt.show()

The bias-adjusted mean lies **above** the median because the log-scale distribution is skewed
on the original scale.

# 3.3 Residual diagnostics

**Fitted values** $\hat{y}_{t|t-1}$ (or simply $\hat{y}_t$) are one-step forecasts of each
observation. They are not always true forecasts: for the mean and drift methods the parameter
($\bar{y}$ or the drift $c$) uses *all* the data, including later observations. For naïve and
seasonal naïve there is no parameter, so fitted values are genuine forecasts.

**Residuals** $e_t = y_t - \hat{y}_t$ are what is left over. A good method gives residuals that:

1. are **uncorrelated** (otherwise information is left in them), and
2. have **zero mean** (otherwise the forecasts are biased — fix by adding the mean).

Useful but not essential: 3. **constant variance** and 4. **normally distributed** — these make
prediction intervals easier. Passing these checks does **not** mean a method can't be improved,
and checking residuals is not a way to *choose* between methods.

## Example: Google daily closing price

For stock prices the naïve method is often best, so the residuals are just the daily changes
$e_t = y_t - y_{t-1}$. The big jump at day 166 (18 Oct 2013, strong quarterly results) shows
up as one large residual.

### The series and its naïve residuals (Figures 3.5–3.8)

In [ ]:
p_ = fpp.autoplot(goog200, title="Google Stock (daily ending 6 December 2013)", xlab="Day",
                  ylab="Closing Price (US$)")
plt.show()
res = fpp.naive(goog200).residuals
p_ = fpp.autoplot(res, title="Residuals from naive method", xlab="Day")
plt.show()
fig, ax = plt.subplots()
p_ = ax.hist(res.dropna(), bins=30)
p_ = ax.set_title("Histogram of residuals")
plt.show()
p_ = fpp.acf_plot(res, title="ACF of residuals")
plt.show()
print({"mean": round(float(res.mean()), 4), "sd": round(float(res.std()), 4)})

The mean is close to zero, there is no significant autocorrelation, but the histogram has a
long right tail (the day-166 jump) — so normal-based intervals may be inaccurate, while the
forecasts themselves are fine.

## Portmanteau tests for autocorrelation

Instead of eyeballing each ACF spike (with 20 spikes, one may cross the bounds by chance),
test the first $h$ autocorrelations **together**:

* **Box-Pierce**: $Q = T \sum_{k=1}^{h} r_k^2$
* **Ljung-Box** (more accurate): $Q^* = T(T+2)\sum_{k=1}^{h} (T-k)^{-1} r_k^2$

Suggested $h = 10$ for non-seasonal data and $h = 2m$ for seasonal data (at most $T/5$).
If the residuals are white noise, $Q$ and $Q^*$ follow a $\chi^2$ distribution with $h - K$
degrees of freedom ($K$ = number of estimated parameters; $K = 0$ for raw data / naïve).
Large values (small p-values) mean the autocorrelations are **not** white noise.

### Box-Pierce and Ljung-Box tests on the naïve residuals

In [ ]:
print(ch3.box_pierce(res, lag=10, df=0).round(4))
print(fpp.ljung_box(res, lag=10, df=0).round(4))

Both p-values are large: the residuals are indistinguishable from white noise.

### All checks at once: `checkresiduals()`

In [ ]:
lb = fpp.checkresiduals(fpp.naive(goog200))

# 3.4 Evaluating forecast accuracy

## Training and test sets

Residuals are **not** a reliable guide to true forecast errors. Judge accuracy on **test data**
that were not used to fit the model:

* test set ≈ 20% of the sample, ideally at least as long as the maximum forecast horizon;
* a good fit to the training data does not imply good forecasts;
* a perfect fit is always possible with enough parameters — **over-fitting** is as bad as missing
  a real pattern.

(Also called hold-out / out-of-sample data.)

### Functions to subset a time series

In [ ]:
print(fpp.window(ausbeer, start=1995).head(8))                 # from 1995 onwards
print(ausbeer.iloc[len(ausbeer) - 4 * 5:])                     # the last 5 years (R's start index is 1-based)
print(ausbeer[ausbeer.index.quarter == 1].head(8))             # all first quarters
print(ausbeer.tail(4 * 5).equals(ausbeer.iloc[-20:]))          # tail() gives the same last 5 years

*Note: R's `subset(start = n - 20)` keeps 21 values (index $n-20$ to $n$) — Python's
`iloc[n-20:]` keeps 20. `tail(x, 20)` is the clean way in both.*

## Forecast errors

A forecast **error** $e_{T+h} = y_{T+h} - \hat{y}_{T+h|T}$ is the unpredictable part of an
observation, measured on the **test** set. Residuals are on the training set and are one-step;
errors can be multi-step.

**Scale-dependent** (same units as the data; can't compare series with different units):
$\text{MAE} = \text{mean}(|e_t|)$, $\text{RMSE} = \sqrt{\text{mean}(e_t^2)}$.
Minimising MAE gives median forecasts; minimising RMSE gives mean forecasts.

**Percentage errors** $p_t = 100 e_t / y_t$ (unit-free): $\text{MAPE} = \text{mean}(|p_t|)$.
Undefined/infinite when $y_t = 0$, extreme near zero, and need a meaningful zero (not
temperatures in °C/°F). They punish negative errors more than positive ones, which led to
$\text{sMAPE} = \text{mean}(200|y_t - \hat{y}_t|/(y_t + \hat{y}_t))$ — but sMAPE can be
negative and is still unstable near zero; the authors recommend **not** using it.

**Scaled errors** (Hyndman & Koehler, 2006): scale by the in-sample MAE of a naïve
(non-seasonal) or seasonal naïve (seasonal) method:
$$q_j = \frac{e_j}{\frac{1}{T-m}\sum_{t=m+1}^{T}|y_t - y_{t-m}|}, \qquad \text{MASE} = \text{mean}(|q_j|).$$
$q_j < 1$: better than the average one-step (seasonal) naïve forecast on the training data.

### Beer: accuracy on 2008–2010 (Figure 3.9)

In [ ]:
beer_fcs = {"Mean": fpp.meanf(beer2, h=10), "Naive": fpp.rwf(beer2, h=10),
            "Seasonal naive": fpp.snaive(beer2, h=10)}
p_ = ch3.plot_forecasts(fpp.window(ausbeer, start=1992), beer_fcs,
                        title="Forecasts for quarterly beer production", xlab="Year", ylab="Megalitres")
plt.show()
beer3 = fpp.window(ausbeer, start=2008)
ch3.accuracy_table(beer_fcs, beer3).round(3)

Seasonal naïve wins on every measure — sometimes the obvious benchmark is hard to beat.

### Google: accuracy on days 201–240 (Figure 3.10)

In [ ]:
goog = fpp.load("goog")
goog_fcs = {"Mean": fpp.meanf(goog200, h=40), "Naive": fpp.rwf(goog200, h=40),
            "Drift": fpp.rwf(goog200, drift=True, h=40)}
g240 = goog.iloc[:240]; g240.attrs = dict(goog.attrs)
p_ = ch3.plot_forecasts(g240, goog_fcs, title="Google stock price (daily ending 6 Dec 13)",
                        xlab="Day", ylab="Closing Price (US$)")
plt.show()
googtest = fpp.window(goog, start=201, end=240)
ch3.accuracy_table(goog_fcs, googtest).round(3)

The drift method is best on every measure here.

## Time series cross-validation

A more sophisticated version of train/test: a series of test sets, each of **one** observation,
with the training set made of all **earlier** observations ("evaluation on a rolling forecasting
origin"). Accuracy = average over the test sets. For $h$-step accuracy the test observation is
$h$ steps after the end of each training set. `tsCV()` returns the matrix of errors.

### Cross-validated RMSE vs residual RMSE

In [ ]:
e = fpp.tsCV(goog200, fpp.rwf, drift=True, h=1)
print(round(float(np.sqrt(np.nanmean(e.to_numpy() ** 2))), 4))                                   # CV RMSE
print(round(float(np.sqrt(np.nanmean(fpp.rwf(goog200, drift=True).residuals.to_numpy() ** 2))), 4))  # residual RMSE

As expected the CV RMSE is larger: residuals are one-step "forecasts" that used all the data.
A good way to choose a model is the **smallest RMSE from time series cross-validation**.

### The pipe operator

R's `%>%` (magrittr) passes the left-hand side as the first argument of the next function, so
code reads left to right. Python's equivalent is **method chaining** or `.pipe()`.

In [ ]:
rmse = lambda x: float(np.sqrt(np.nanmean(np.asarray(x) ** 2)))
print(goog200.pipe(fpp.tsCV, fpp.rwf, drift=True, h=1).pipe(rmse))
print(goog200.pipe(fpp.rwf, drift=True).residuals.pipe(rmse))

### Example: MSE by horizon for 1- to 8-step naïve forecasts (Figure 3.13)

In [ ]:
e8 = fpp.tsCV(goog200, fpp.naive, h=8)
mse = np.nanmean(e8.to_numpy() ** 2, axis=0)
fig, ax = plt.subplots(figsize=(6, 3.5))
p_ = ax.scatter(np.arange(1, 9), mse)
p_ = ax.set(xlabel="h", ylabel="MSE")
plt.show()

Forecast error grows with the horizon, as expected.

# 3.5 Prediction intervals

Assuming normally distributed errors, a 95% interval for the $h$-step forecast is
$\hat{y}_{T+h|T} \pm 1.96\,\hat{\sigma}_h$; in general $\hat{y}_{T+h|T} \pm c\,\hat{\sigma}_h$
with multiplier $c$ from the normal distribution (Table 3.1).

### Multipliers for prediction intervals (Table 3.1)

In [ ]:
from scipy.stats import norm
pct = np.array([50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 96, 97, 98, 99])
pd.DataFrame({"Percentage": pct, "Multiplier": norm.ppf(0.5 + pct / 200).round(2)})

Point forecasts without intervals carry almost no information about how much to trust them.

## One-step prediction intervals

For one step ahead, the forecast standard deviation ≈ the residual standard deviation
(exactly equal when nothing is estimated, as with naïve). For `goog200`: last value 531.48,
residual sd 6.21 → 95% PI $531.48 \pm 1.96(6.21) = [519.3, 543.6]$, 80% PI
$531.48 \pm 1.28(6.21) = [523.5, 539.4]$.

### Computing the one-step intervals by hand

In [ ]:
last = goog200.iloc[-1]; s = fpp.naive(goog200).residuals.std()
pd.DataFrame({"lo": [last - 1.96 * s, last - 1.28 * s], "hi": [last + 1.96 * s, last + 1.28 * s]},
             index=["95%", "80%"]).round(2)

## Multi-step prediction intervals

Intervals usually **widen** with the horizon. For the benchmark methods (assuming uncorrelated
residuals, $\hat\sigma$ = residual sd, $T$ = sample size):

| Method | $\hat{\sigma}_h$ |
|---|---|
| Mean | $\hat\sigma\sqrt{1 + 1/T}$ |
| Naïve | $\hat\sigma\sqrt{h}$ |
| Seasonal naïve | $\hat\sigma\sqrt{k+1}$, $k$ = integer part of $(h-1)/m$ |
| Drift | $\hat\sigma\sqrt{h(1 + h/T)}$ |

For $h = 1$ and large $T$ all are ≈ $\hat\sigma$.

### Naïve forecasts with intervals for `goog200` (Figure 3.14)

In [ ]:
fcn = fpp.naive(goog200)
print(fcn.summary().round(2))
p_ = fpp.plot_forecast(fcn)
plt.show()

## Prediction intervals from bootstrapped residuals

If normality is doubtful but the residuals are uncorrelated with constant variance, simulate
future paths: $y^*_{T+1} = \hat{y}_{T+1|T} + e^*_{T+1}$ where $e^*$ is drawn at random from the
past residuals; repeat step by step, many times, and take percentiles at each horizon —
a **bootstrapped** prediction interval.

### Bootstrapped naïve intervals

*(Simulation-based: R and Python numbers differ slightly from run to run.)*

In [ ]:
ch3.naive_bootstrap(goog200, h=10, npaths=5000, seed=2018).summary().round(1)

Very similar (but not identical) to the normal-based intervals.

## Prediction intervals with transformations

Compute the interval on the transformed scale and back-transform its end points. Coverage is
preserved, but the interval is no longer symmetric around the point forecast. With the
`lambda` argument (`lambda_` in `fpp.py`) this is done automatically — see Figure 3.4 above.

# 3.6 The forecast package in R

Functions that return a `forecast` object: `meanf()`, `naive()`, `snaive()`, `rwf()`,
`croston()`, `stlf()`, `ses()`, `holt()`, `hw()`, `splinef()`, `thetaf()`, `forecast()`.
Every `forecast` object can be printed, summarised and plotted the same way.

The usual workflow later in the book: **fit a model, then call `forecast()`** on it. Called
directly on a time series, `forecast()` uses the automatic ETS algorithm of Chapter 7 —
useful when you have no idea what model to use, but by the end of the book you should choose
the model yourself.

### `forecast()` on a raw series

In [ ]:
from statsforecast.models import AutoETS
m = AutoETS(season_length=4).fit(ausbeer.to_numpy())
p = m.predict(h=4, level=[80, 95])
print("Model:", m.model_["method"])
pd.DataFrame({"Point Forecast": p["mean"], "Lo 80": p["lo-80"], "Hi 80": p["hi-80"],
              "Lo 95": p["lo-95"], "Hi 95": p["hi-95"]},
             index=pd.period_range(ausbeer.index[-1] + 1, periods=4, freq="Q")).round(1)

# Model testing — every model used in this chapter

Every method the chapter teaches is tested **out of sample**:

| Group | Models | Data (train → test) |
|---|---|---|
| Benchmarks | mean, naïve, seasonal naïve, drift | `ausbeer` 1992–2007 → 2008–10; `goog200` → days 201–240; `a10` to 2005 → 2006–08 |
| Transformations | seasonal naïve and drift, each with no transformation, log (λ = 0) and Guerrero λ | `a10` |
| Bias adjustment | drift on log scale, median vs bias-adjusted mean | `eggs` 1900–1983 → 1984–93 |
| Intervals | normal vs bootstrapped naïve intervals: empirical 80%/95% coverage | `goog200` → 201–240 |
| Cross-validation | tsCV RMSE (h = 1) of mean, naïve, drift | `goog200` |
| `forecast()` default | automatic ETS | `ausbeer` 1992–2007 → 2008–10 |

### Test-set accuracy of all methods

In [ ]:
beer_tr = fpp.window(ausbeer, start=1992, end=(2007, 4)); beer_te = fpp.window(ausbeer, start=2008)
g_te = fpp.window(goog, start=201, end=240)
a10 = fpp.load("a10")
a10_tr = fpp.window(a10, end=(2005, 12)); a10_te = fpp.window(a10, start=2006)
eggs_tr = fpp.window(eggs, end=1983); eggs_te = fpp.window(eggs, start=1984)
lam_a10 = fpp.box_cox_lambda(a10_tr)
hb, ha, he = len(beer_te), len(a10_te), len(eggs_te)

ets_beer = AutoETS(season_length=4).fit(beer_tr.to_numpy()).predict(h=hb)["mean"]
cols = ["RMSE", "MAE", "MAPE", "MASE"]
acc = lambda fc, x: fpp.accuracy(fc, x).loc["Test set", cols]
results_py = pd.DataFrame({
    "beer: mean":                acc(fpp.meanf(beer_tr, h=hb), beer_te),
    "beer: naive":               acc(fpp.naive(beer_tr, h=hb), beer_te),
    "beer: seasonal naive":      acc(fpp.snaive(beer_tr, h=hb), beer_te),
    "beer: drift":               acc(fpp.rwf(beer_tr, drift=True, h=hb), beer_te),
    "beer: AutoETS":             fpp.accuracy(ets_beer, beer_te, train=beer_tr).loc["Test set", cols],
    "goog: mean":                acc(fpp.meanf(goog200, h=40), g_te),
    "goog: naive":               acc(fpp.naive(goog200, h=40), g_te),
    "goog: drift":               acc(fpp.rwf(goog200, drift=True, h=40), g_te),
    "a10: seasonal naive":       acc(fpp.snaive(a10_tr, h=ha), a10_te),
    "a10: snaive, log":          acc(fpp.snaive(a10_tr, h=ha, lambda_=0), a10_te),
    "a10: snaive, Box-Cox":      acc(fpp.snaive(a10_tr, h=ha, lambda_=lam_a10), a10_te),
    "a10: drift":                acc(fpp.rwf(a10_tr, drift=True, h=ha), a10_te),
    "a10: drift, log":           acc(fpp.rwf(a10_tr, drift=True, h=ha, lambda_=0), a10_te),
    "a10: drift, Box-Cox":       acc(fpp.rwf(a10_tr, drift=True, h=ha, lambda_=lam_a10), a10_te),
    "eggs: drift, log (median)": acc(fpp.rwf(eggs_tr, drift=True, lambda_=0, h=he), eggs_te),
    "eggs: drift, log (mean)":   acc(fpp.rwf(eggs_tr, drift=True, lambda_=0, h=he, biasadj=True), eggs_te),
}).T
results_py.round(3)

### Prediction-interval coverage and cross-validation

In [ ]:
fn = fpp.naive(goog200, h=40); fb = ch3.naive_bootstrap(goog200, h=40, seed=2018)
fs = fpp.snaive(beer_tr, h=hb)
print(pd.DataFrame({"model": ["goog naive (normal)", "goog naive (bootstrap)", "beer snaive (normal)"],
                    "cover80": [ch3.coverage(fn, g_te, 80), ch3.coverage(fb, g_te, 80), ch3.coverage(fs, beer_te, 80)],
                    "cover95": [ch3.coverage(fn, g_te, 95), ch3.coverage(fb, g_te, 95), ch3.coverage(fs, beer_te, 95)]}))
cv_rmse = lambda f, **kw: float(np.sqrt(np.nanmean(fpp.tsCV(goog200, f, h=1, **kw).to_numpy() ** 2)))
print({"mean": round(cv_rmse(fpp.meanf), 4), "naive": round(cv_rmse(fpp.naive), 4),
       "drift": round(cv_rmse(fpp.rwf, drift=True), 4)})

### What the tests show

* **Beer:** seasonal naïve is the best benchmark (MASE well below the others); the automatic
  ETS model from `forecast()` beats it slightly — a "real" model has to earn its place against
  this benchmark.
* **Google:** drift wins on the 40-day test set, but in **cross-validation** (1-step) naïve and
  drift are almost identical and both crush the mean method — the stock is close to a random walk.
* **Transformations:** seasonal naïve point forecasts are **identical** with or without a
  transformation (it just repeats last year's values, and back-transforming undoes the transform) —
  only the intervals change, which illustrates the Section 3.2 remark. For drift the transformation
  does change the forecasts (on the log scale the drift becomes a constant *growth rate*), but
  here the untransformed drift scored best on 2006–08: stabilising the variance does not
  automatically improve point accuracy — always test. (None of these benchmarks captures both the
  trend and the seasonality of `a10`; the models of Chapters 6–8 do.)
* **Bias adjustment:** for `eggs` the bias-adjusted (mean) forecasts are slightly higher than
  the median forecasts; which scores better depends on the measure (MAE favours medians).
* **Intervals:** coverage on short test sets is noisy, but normal and bootstrap intervals give
  similar coverage, as the book notes.
* **R vs Python:** all benchmark forecasts, intervals, accuracy measures, Box-Cox λ and tsCV
  results are **identical** (they use the same formulas; see `tests/python/test_fpp_core.py` and
  `tests/python/test_ch03.py`). Only the bootstrap intervals (random) and the automatic ETS
  choice (different optimisers) can differ.

# Key takeaways

* Always compare against **mean, naïve, seasonal naïve and drift**.
* Adjust/transform to simplify patterns; back-transform (and bias-adjust if you need means).
* Good residuals: **uncorrelated, zero mean** (nice to have: constant variance, normal) — test with Ljung-Box.
* Judge accuracy on **test data** or with **time series cross-validation**; prefer MASE/RMSE/MAE over MAPE/sMAPE.
* Always give **prediction intervals**; they widen with the horizon.

*Reference: Hyndman, R.J., & Athanasopoulos, G. (2018). Forecasting: Principles and Practice
(2nd ed.). OTexts: Melbourne, Australia. <https://otexts.com/fpp2/>*